In [19]:
import os
import pickle
import re
import gc
from pathlib import Path

import pandas as pd
import numpy as np
from utils.model_saver import *

PROJECT_ROOT = Path().resolve().parent.parent
print(f"Project root: {PROJECT_ROOT}")

PATH_DATA = PROJECT_ROOT / 'data'
PATH_NORMAL_DATA = PATH_DATA / 'normal_features'
PATH_GRAPH_DATA = PATH_DATA / 'graph_features'
PATH_TEXT_DATA = PATH_DATA / 'textual_features'
PATH_COMBINED_DATA = PATH_DATA / 'combined_features'
MODELS_SAVE_PATH = PROJECT_ROOT / 'Models'

SAVE = True
PATH_PLOTS = PROJECT_ROOT / 'report' / 'src'
TARGET = 'is_reference_valid'
RANDOM_STATE = 42

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Keep predictions memory-friendly. Increase manually only if you have enough RAM.
N_JOBS = 1


Project root: C:\Users\ivan.bernacchia\OneDrive - TINEXT SA\Documents\_Alessia\M-P6203E-DataProjects-Hackaton3_P1
Using device: cpu


In [11]:
ID_COLUMNS = ["article_id", "ref_id"]

FEATURE_GROUPS = [
    {"model_key": "initial_features", "name_tag": "Normal", "slug": "normal", "train_path": PATH_NORMAL_DATA / "train.parquet", "test_path": PATH_NORMAL_DATA / "test.parquet"},
    {"model_key": "graph_features", "name_tag": "Graph", "slug": "graph", "train_path": PATH_GRAPH_DATA / "final_train.parquet", "test_path": PATH_GRAPH_DATA / "final_test.parquet"},
    {"model_key": "textual_embeddings_64", "name_tag": "Textual 64", "slug": "textual_64", "single_path": PATH_TEXT_DATA / "textual_embeddings_64.parquet"},
    {"model_key": "textual_embeddings_128", "name_tag": "Textual 128", "slug": "textual_128", "single_path": PATH_TEXT_DATA / "textual_embeddings_128.parquet"},
    {"model_key": "all_features", "name_tag": "Mix", "slug": "mix", "train_path": PATH_COMBINED_DATA / "train.parquet", "test_path": PATH_COMBINED_DATA / "test.parquet"},
]
FEATURE_GROUP_BY_KEY = {group["model_key"]: group for group in FEATURE_GROUPS}

def _drop_id_columns(df):
    return df.drop(columns=ID_COLUMNS, errors="ignore")

def _read_split_from_single_parquet(path, split_name):
    try:
        df = pd.read_parquet(path, filters=[("split", "==", split_name)])
        if not df.empty:
            return df
    except Exception:
        pass
    return None

def load_feature_dataset(group):
    """Load only one feature dataset pair, then return train/test DataFrames."""
    if "single_path" in group:
        path = group["single_path"]
        train_df = _read_split_from_single_parquet(path, "train")
        test_df = _read_split_from_single_parquet(path, "test")

        if train_df is None or test_df is None:
            df = pd.read_parquet(path)
            split_series = df["split"].astype(str).str.lower()
            train_df = df.loc[split_series == "train"].copy()
            test_df = df.loc[split_series == "test"].copy()
            del df, split_series

        train_df = _drop_id_columns(train_df)
        test_df = _drop_id_columns(test_df)
    else:
        train_df = _drop_id_columns(pd.read_parquet(group["train_path"]))
        test_df = _drop_id_columns(pd.read_parquet(group["test_path"]))
    gc.collect()
    return train_df, test_df


In [30]:
import pickle
import torch
from pathlib import Path
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

def load_latest_models(base_path='./Models', device='cuda'):
    """
    Creates a nested dictionary registry.
    Access via: registry['folder_name']['KNN']
    """
    models_registry = {}
    base_dir = Path(base_path)
    if not base_dir.exists():
        print(f"Error: Directory {base_path} does not exist.")
        return model_paths
    model_types = {'KNN': 'Best_KNN', 'XGB': 'Best_XGB', 'transformer': 'Transformer'}
    for subdir in base_dir.iterdir():
        if not subdir.is_dir():
            continue
        model_paths[subdir.name] = {}
        for label, keyword in model_types.items():
            model_files = [f for f in subdir.glob(f'*{keyword}*.pkl')]
            if model_files:
                model_paths[subdir.name][label] = sorted(model_files, key=lambda x: x.name)[-1]
    return model_paths

                # Sort alphabetically to get the latest timestamp at the end
                latest_file = sorted(model_files, key=lambda x: x.name)[-1]
                
                try:
                    with open(latest_file, 'rb') as f:
                        if label == 'transformer':
                            safe_device = 'cuda' if (device == 'cuda' and torch.cuda.is_available()) else 'cpu'
                            # pytorch specific import to respect the device (cpu or cuda)
                            print(f"Loading transformer on device: {safe_device}")
                            models_registry[subdir.name][label] = torch.load(f, 
                                                                             map_location=torch.device(safe_device),
                                                                             weights_only=False)
                        else:
                            # standard pickle load
                            models_registry[subdir.name][label] = pickle.load(f)
                    print(f"Loaded {label} from {subdir.name}: {latest_file.name}")
                except Exception as e:
                    print(f"Failed to load {label} in {subdir.name}: {e}")

    return models_registry

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

def evaluate_model_on_sets(model, set_dict):
    results = {'cms': {}, 'metrics': []}
    for set_name, (X, y) in set_dict.items():
        y_pred = model.predict(X)
        results['cms'][set_name] = confusion_matrix(y, y_pred)
        results['metrics'].append({
            'Set': set_name,
            'F1': f1_score(y, y_pred, average='weighted'),
            'Accuracy': accuracy_score(y, y_pred),
            'Precision': precision_score(y, y_pred, average='weighted'),
            'Recall': recall_score(y, y_pred, average='weighted')
        })
        del y_pred
    results['metrics_df'] = pd.DataFrame(results['metrics']).set_index('Set')
    return results

def plot_model_comparison(model_list, model_names, all_sets_list, title="Model Comparison", figsize=(18, 10)):
    n_models = len(model_list)
    set_names = list(all_sets_list[0].keys())
    n_sets = len(set_names)
    fig, axes = plt.subplots(n_sets, n_models + 1, figsize=figsize, constrained_layout=True)
    axes = np.asarray(axes).reshape(n_sets, n_models + 1)
    fig.suptitle(title, fontsize=20, fontweight='bold')
    all_evaluations = []
    metrics_rows = []
    for i, (model, m_name, s_dict) in enumerate(zip(model_list, model_names, all_sets_list)):
        res = evaluate_model_on_sets(model, s_dict)
        all_evaluations.append(res)
        for set_name, row in res['metrics_df'].iterrows():
            row_dict = row.to_dict()
            row_dict['Model_Type'] = m_name
            row_dict['Set'] = set_name
            metrics_rows.append(row_dict)
        for j, s_name in enumerate(set_names):
            ax = axes[j, i]
            sns.heatmap(res['cms'][s_name], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
            ax.set_title(f"{m_name} - {s_name}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
    for j, s_name in enumerate(set_names):
        ax = axes[j, -1]
        stats_data = []
        for idx, m_name in enumerate(model_names):
            m_metrics = all_evaluations[idx]['metrics_df'].loc[s_name]
            for metric_name in ['F1', 'Accuracy']:
                stats_data.append({'Model': m_name, 'Metric': metric_name, 'Score': m_metrics[metric_name]})
        df_plot = pd.DataFrame(stats_data)
        sns.barplot(data=df_plot, x='Score', y='Metric', hue='Model', ax=ax)
        ax.set_xlim(0, 1.0)
        ax.set_title(f"Global Stats - {s_name}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    metrics_df = pd.DataFrame(metrics_rows).set_index(['Model_Type', 'Set'])
    return fig, metrics_df

def separe_dataset(dataset, target='is_reference_valid'):
    X = dataset.drop(columns=[target], errors='raise')
    y = dataset[target]
    return X, y

def set_dict(datasets, names=['train', 'test'], target='is_reference_valid'):
    return {name: separe_dataset(df, target=target) for df, name in zip(datasets, names)}

def run_feature_comparison(group, model_paths, figsize=(20, 16), close_after_save=True):
    """Load one dataset + its models, evaluate them, save tiny metrics, then free RAM."""
    name_tag = group['name_tag']
    print(f"\n=== {name_tag} ===")
    train_df, test_df = load_feature_dataset(group)
    sets_for_group = set_dict([train_df, test_df], names=['train', 'test'], target=TARGET)
    group_models = load_models_for_group(model_paths, group['model_key'])
    ordered_labels = ['KNN', 'XGB', 'transformer']
    model_list = [group_models[label] for label in ordered_labels if label in group_models]
    model_names = [f"{label.upper() if label != 'transformer' else 'Transformer'} ({name_tag})" for label in ordered_labels if label in group_models]
    if not model_list:
        raise ValueError(f"No models found for group {group['model_key']}")
    fig, metrics_df = plot_model_comparison(
        model_list=model_list,
        model_names=model_names,
        all_sets_list=[sets_for_group] * len(model_list),
        title=f"{name_tag} Features - Evaluation Results",
        figsize=figsize,
    )
    output_path = PATH_PLOTS / f"eval_models_{group['slug']}.png"
    if SAVE:
        fig.savefig(output_path, bbox_inches='tight')
        print(f"Saved figure: {output_path}")
    plt.show()
    if close_after_save:
        plt.close(fig)
    del train_df, test_df, sets_for_group, group_models, model_list
    gc.collect()
    return metrics_df, output_path


# Models Comparison


## 1. Import Pretrained Models

In [31]:
models_dict = load_latest_models(MODELS_SAVE_PATH, device=DEVICE)

Loaded KNN from graph_features: Best_KNN_graph_based_20260430_150722.pkl
Loaded XGB from graph_features: Best_XGB_graph_20260430_152157.pkl
Loading transformer on device: cpu
Failed to load transformer in graph_features: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.


C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:07:40] WARNING: D:\bld\xgboost-split_1772124962567\work\src\gbm\gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  models_registry[subdir.name][label] = pickle.load(f)
C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:07:40] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  models_registry[subdir.name][label] = pickle.load(f)


Loaded KNN from initial_features: Best_KNN_initial_based_20260502_144830.pkl
Loaded XGB from initial_features: Best_XGB_initial_based_20260502_145227.pkl
Loading transformer on device: cpu
Failed to load transformer in initial_features: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.
Loaded XGB from textual_embeddings_128: Best_XGB_textual_128_20260430_152616.pkl
Loading transformer on device: cpu
Failed to load transformer in textual_embeddings_128: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.
Loaded XGB from textual_embeddings_64: Best_XGB_textual_64_20260430_144945.pkl
Loading transformer on device: cpu
Failed to load transformer in

In [32]:
# Lightweight registry: paths only, no models loaded yet.
model_paths = find_latest_model_paths(MODELS_SAVE_PATH)
comparison_metrics = {}
comparison_figure_paths = {}


Loaded KNN from graph_features: Best_KNN_graph_based_20260430_150722.pkl
Loaded XGB from graph_features: Best_XGB_graph_20260430_152157.pkl
Loading transformer on device: cpu
Failed to load transformer in graph_features: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.


C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:11:06] WARNING: D:\bld\xgboost-split_1772124962567\work\src\gbm\gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  models_registry[subdir.name][label] = pickle.load(f)
C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:11:06] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  models_registry[subdir.name][label] = pickle.load(f)


Loaded KNN from initial_features: Best_KNN_initial_based_20260502_144830.pkl
Loaded XGB from initial_features: Best_XGB_initial_based_20260502_145227.pkl
Loading transformer on device: cpu
Failed to load transformer in initial_features: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.
Loaded XGB from textual_embeddings_128: Best_XGB_textual_128_20260430_152616.pkl
Loading transformer on device: cpu
Failed to load transformer in textual_embeddings_128: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.


C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:11:07] WARNING: D:\bld\xgboost-split_1772124962567\work\src\gbm\gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  models_registry[subdir.name][label] = pickle.load(f)
C:\Users\ivan.bernacchia\AppData\Local\Temp\ipykernel_26168\3239983947.py:51: UserWarning: [22:11:07] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  models_registry[subdir.name][label] = pickle.load(f)


Loaded XGB from textual_embeddings_64: Best_XGB_textual_64_20260430_144945.pkl
Loading transformer on device: cpu
Failed to load transformer in textual_embeddings_64: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.


KeyError: 'transformer'

## 2. Comparison by feature types

### 2.1 Initial Features

In [ ]:
metrics_df, fig_path = run_feature_comparison(FEATURE_GROUP_BY_KEY['initial_features'], model_paths)
comparison_metrics['initial_features'] = metrics_df
comparison_figure_paths['initial_features'] = fig_path


### 2.2 Graph Features

In [ ]:
metrics_df, fig_path = run_feature_comparison(FEATURE_GROUP_BY_KEY['graph_features'], model_paths)
comparison_metrics['graph_features'] = metrics_df
comparison_figure_paths['graph_features'] = fig_path


### 2.3 Textual features

In [ ]:
metrics_df, fig_path = run_feature_comparison(FEATURE_GROUP_BY_KEY['textual_embeddings_64'], model_paths)
comparison_metrics['textual_embeddings_64'] = metrics_df
comparison_figure_paths['textual_embeddings_64'] = fig_path


In [ ]:
metrics_df, fig_path = run_feature_comparison(FEATURE_GROUP_BY_KEY['textual_embeddings_128'], model_paths)
comparison_metrics['textual_embeddings_128'] = metrics_df
comparison_figure_paths['textual_embeddings_128'] = fig_path


### 2.4 Combined features

In [ ]:
metrics_df, fig_path = run_feature_comparison(FEATURE_GROUP_BY_KEY['all_features'], model_paths)
comparison_metrics['all_features'] = metrics_df
comparison_figure_paths['all_features'] = fig_path


## 3. Global Comparison

### 3.1 All figures

In [ ]:
# Figures are saved and closed after each group to avoid keeping them in RAM.
pd.Series(comparison_figure_paths, name='saved_figure')


> TODO comments

### 3.2 Heatmap of the stats

In [ ]:
# Heatmap uses the stored metrics only; datasets and models have already been released.


In [ ]:
def plot_global_stats_heatmap(metrics_by_group, figsize=(12, 10)):
    """Collects test stats from all groups and plots a global heatmap."""
    if not metrics_by_group:
        raise ValueError("No metrics available. Run the feature comparison cells first.")
    df_stats = pd.concat(metrics_by_group.values()).reset_index()
    df_stats = df_stats[df_stats['Set'] == 'test'].set_index('Model_Type')
    df_stats = df_stats[['F1', 'Accuracy', 'Precision', 'Recall']]
    plt.figure(figsize=figsize)
    sns.heatmap(df_stats, annot=True, cmap='YlGnBu', fmt='.4f', cbar_kws={'label': 'Score'})
    plt.title("Global Model Comparison - Test Set Metrics", fontsize=16, fontweight='bold')
    plt.ylabel("Model (Feature Set)")
    plt.tight_layout()
    if SAVE:
        output_path = PATH_PLOTS / 'global_stats_heatmap.png'
        plt.savefig(output_path, bbox_inches='tight')
        print(f"Saved figure: {output_path}")
    plt.show()
    plt.close()
    return df_stats


In [ ]:
global_test_metrics = plot_global_stats_heatmap(comparison_metrics)
global_test_metrics


> TODO comments